# GCS Connection Test

Verify that `google-cloud-storage` can authenticate and reach the scBaseCount bucket.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv
from google.cloud import storage as gcs

load_dotenv(dotenv_path=Path("..") / ".env")

GCP_PROJECT = os.environ["GCP_PROJECT"]
H5AD_PATH = os.environ["H5AD_PATH"]  # gs://arc-institute-virtual-cell-atlas/...
META_PATH = os.environ["META_PATH"]

## 1. Check credentials

If this cell raises `google.auth.exceptions.DefaultCredentialsError`, run:
```bash
gcloud auth application-default login
```

In [ ]:
import google.auth

credentials, project = google.auth.default()
print("Credentials type:", type(credentials).__name__)
print("Project:         ", project or "(none set)")

## 2. List a few objects in the h5ad prefix

In [ ]:
parsed = urlparse(H5AD_PATH)
bucket_name = parsed.netloc
prefix = parsed.path.lstrip("/") + "/"

client = gcs.Client(project=GCP_PROJECT)
blobs = list(client.list_blobs(bucket_name, prefix=prefix, max_results=10))

print(f"Bucket : {bucket_name}")
print(f"Prefix : {prefix}")
print(f"First {len(blobs)} object(s):")
for b in blobs:
    size_mb = b.size / 1024 / 1024
    print(f"  gs://{bucket_name}/{b.name}  ({size_mb:.1f} MB)")

## 3. Download a single file

Pick the first blob from the listing above and download it to a temp path to confirm read access.

In [ ]:
import tempfile

if not blobs:
    print("No blobs found; skipping download test.")
else:
    target_blob = blobs[0]
    with tempfile.NamedTemporaryFile(suffix=".h5ad", delete=False) as tmp:
        tmp_path = Path(tmp.name)

    print(f"Downloading {target_blob.name} -> {tmp_path}")
    target_blob.download_to_filename(str(tmp_path))
    size_mb = tmp_path.stat().st_size / 1024 / 1024
    print(f"Downloaded {size_mb:.1f} MB")

    # Clean up
    tmp_path.unlink()
    print("Temp file deleted.")